In [1]:
import pandas as pd
import numpy as np



In [8]:
movies = pd.read_csv("./data/movies.csv")
ratings = pd.read_csv("./data/ratings.csv")
users = pd.read_csv("./data/users.csv")

In [9]:
print(movies.shape)
print(ratings.shape)
print(users.shape)

(3706, 4)
(1000209, 4)
(6040, 6)


In [10]:
movies.head()

,movie_id,movie_title,movie_genres,poster_url
0,3107,Backdraft (1991),"0, 7",https://image.tmdb.org/t/p/w500/bymJtYNIdiAiBR...
1,2114,"Outsiders, The (1983)",7,https://image.tmdb.org/t/p/w500/pl8Tf36TAOb2i5...
2,256,Junior (1994),"4, 15",https://image.tmdb.org/t/p/w500/4tRF43CbEYHmV8...
3,1389,Jaws 3-D (1983),"0, 10",https://image.tmdb.org/t/p/w500/kqDXj53F9paqVG...
4,3635,"Spy Who Loved Me, The (1977)",0,https://image.tmdb.org/t/p/w500/3ZxHKFxMYvAko6...


In [11]:
movies.isnull().sum()

movie_id         0
movie_title      0
movie_genres     0
poster_url      58
dtype: int64

In [12]:
ratings.isnull().sum()

user_id        0
movie_id       0
user_rating    0
timestamp      0
dtype: int64

In [13]:
users.isnull().sum()

user_id                  0
user_gender              0
bucketized_user_age      0
user_occupation_label    0
user_occupation_text     0
user_zip_code            0
dtype: int64

In [14]:
ratings.duplicated().sum()

np.int64(0)

In [15]:
ratings.groupby(['user_id','movie_id']).size().max()

np.int64(1)

In [16]:
#Popularity-based-model

In [17]:
avg_rating=ratings.groupby('movie_id')['user_rating'].mean()

In [18]:
count_rating=ratings.groupby('movie_id')['user_rating'].count()

In [19]:
count_rating

movie_id
1       2077
2        701
3        478
4        170
5        296
        ... 
3948     862
3949     304
3950      54
3951      40
3952     388
Name: user_rating, Length: 3706, dtype: int64

In [20]:
count_rating.sort_values(ascending=False).head(10)

movie_id
2858    3428
260     2991
1196    2990
1210    2883
480     2672
2028    2653
589     2649
2571    2590
1270    2583
593     2578
Name: user_rating, dtype: int64

In [21]:
movies_state=pd.DataFrame({
    "avg_rating":avg_rating,
    "count_rating":count_rating
}).reset_index()

In [22]:
movies_state['count_rating'].max()

np.int64(3428)

In [23]:
c=movies_state['avg_rating'].mean()
m=movies_state['count_rating'].quantile(0.90)

In [24]:
c

np.float64(3.2388921779108912)

In [25]:
m

np.float64(729.5)

In [26]:
movies_state["weighted_score"] = (
    (movies_state["count_rating"] /
     (movies_state["count_rating"] + m))
    * movies_state["avg_rating"]
    +
    (m /
     (movies_state["count_rating"] + m))
  *c)

In [27]:
movies_state

,movie_id,avg_rating,count_rating,weighted_score
0,1,4.146846,2077,3.910840
1,2,3.201141,701,3.220393
2,3,3.016736,478,3.150950
3,4,2.729412,170,3.142603
4,5,3.006757,296,3.171889
...,...,...,...,...
3701,3948,3.635731,862,3.453831
3702,3949,4.115132,304,3.496635
3703,3950,3.666667,54,3.268375
3704,3951,3.900000,40,3.273258


In [28]:
popular_movies=movies_state.sort_values('weighted_score',ascending=False)

In [29]:
popular_movies

,movie_id,avg_rating,count_rating,weighted_score
309,318,4.554558,2227,4.229925
253,260,4.453694,2991,4.215501
802,858,4.524966,2223,4.207205
513,527,4.510417,2304,4.204639
1108,1198,4.477725,2514,4.199097
...,...,...,...,...
532,546,1.874286,350,2.796454
1435,1562,2.257426,606,2.793539
1432,1556,1.871935,367,2.781370
3353,3593,1.611111,342,2.719339


In [30]:
popular_movies.head(10)

,movie_id,avg_rating,count_rating,weighted_score
309,318,4.554558,2227,4.229925
253,260,4.453694,2991,4.215501
802,858,4.524966,2223,4.207205
513,527,4.510417,2304,4.204639
1108,1198,4.477725,2514,4.199097
49,50,4.517106,1783,4.145979
2557,2762,4.406263,2459,4.139179
2651,2858,4.317386,3428,4.128147
579,593,4.351823,2578,4.106356
1848,2028,4.337354,2653,4.100450


In [31]:
movies.head()

,movie_id,movie_title,movie_genres,poster_url
0,3107,Backdraft (1991),"0, 7",https://image.tmdb.org/t/p/w500/bymJtYNIdiAiBR...
1,2114,"Outsiders, The (1983)",7,https://image.tmdb.org/t/p/w500/pl8Tf36TAOb2i5...
2,256,Junior (1994),"4, 15",https://image.tmdb.org/t/p/w500/4tRF43CbEYHmV8...
3,1389,Jaws 3-D (1983),"0, 10",https://image.tmdb.org/t/p/w500/kqDXj53F9paqVG...
4,3635,"Spy Who Loved Me, The (1977)",0,https://image.tmdb.org/t/p/w500/3ZxHKFxMYvAko6...


In [32]:
popular_movies=popular_movies.merge(
    movies[['movie_id','movie_title','movie_genres','poster_url']],
    on='movie_id',
    how='left'
)

In [33]:
popular_movies.head()

,movie_id,avg_rating,count_rating,weighted_score,movie_title,movie_genres,poster_url
0,318,4.554558,2227,4.229925,"Shawshank Redemption, The (1994)",7,https://image.tmdb.org/t/p/w500/9cqNxx0GxF0bfl...
1,260,4.453694,2991,4.215501,Star Wars: Episode IV - A New Hope (1977),"0, 1, 8, 15",https://image.tmdb.org/t/p/w500/6FfCtAuVAW8XJj...
2,858,4.524966,2223,4.207205,"Godfather, The (1972)","0, 5, 7",https://image.tmdb.org/t/p/w500/3bhkrj58Vtu7en...
3,527,4.510417,2304,4.204639,Schindler's List (1993),"7, 18",https://image.tmdb.org/t/p/w500/sF1U4EUQS8YHUY...
4,1198,4.477725,2514,4.199097,Raiders of the Lost Ark (1981),"0, 1",https://image.tmdb.org/t/p/w500/ceG9VzoRAVGwiv...


In [34]:
#Colaborative-Model

In [35]:
!pip install scikit-surprise

  Using cached scipy-1.18.1-cp314-cp314-win_amd64.whl.metadata (61 kB)
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   --------------- ------------------------ 0.5/1.3 MB 3.1 MB/s eta 0:00:01
   ----------------------- ---------------- 0.8/1.3 MB 1.5 MB/s eta 0:00:01
   ----------------------- ---------------- 0.8/1.3 MB 1.5 MB/s eta 0:00:01
   ------------------------------- -------- 1.0/1.3 MB 1.2 MB/s eta 0:00:01
   ------------------------------- -------- 1.0/1.3 MB 1.2 MB/s eta 0:00:01
   ---------------------------------------- 1.3/1.3 MB 1.0 MB/s  0:00:01
   ---------------------------------------- 0.0/37.4 MB ? eta -:--:--
   ---------------------------------------- 0.3/37.4 MB ? eta -:--:--
    --------------------------------------- 0.5/37.4 MB 1.6 MB/s eta 0:00:24
    --------------------------------------- 0.8/37.4 MB 1.8 MB/s eta 0:00:21
   - -------------------------------------- 1.0/37.4 MB 1.2 MB/s eta 0:00:31
   - -------------------------------

In [36]:
from surprise import Dataset, Reader, SVD

In [37]:
ratings=ratings.sort_values(['user_id','timestamp'])

In [38]:
ratings

,user_id,movie_id,user_rating,timestamp
486815,1,3186,4.0,978300019
17504,1,1270,5.0,978300055
918576,1,1022,5.0,978300055
985165,1,1721,4.0,978300055
52019,1,2340,3.0,978300103
...,...,...,...,...
940072,6040,2917,4.0,997454429
610722,6040,1921,4.0,997454464
624387,6040,1784,3.0,997454464
338465,6040,161,3.0,997454486


In [39]:
user_counts = ratings.groupby("user_id").size()

In [40]:
user_counts

user_id
1        53
2       129
3        51
4        21
5       198
       ... 
6036    888
6037    202
6038     20
6039    123
6040    341
Length: 6040, dtype: int64

In [41]:
eligible_users = user_counts[user_counts >= 50].index

In [42]:
eligible_users

Index([   1,    2,    3,    5,    6,    8,    9,   10,   11,   13,
       ...
       6026, 6030, 6031, 6032, 6033, 6035, 6036, 6037, 6039, 6040],
      dtype='int64', name='user_id', length=4297)

In [43]:
test = ratings[
    ratings["user_id"].isin(eligible_users)
].groupby("user_id").tail(1)

In [44]:
test

,user_id,movie_id,user_rating,timestamp
743178,1,48,5.0,978824351
722977,2,1917,3.0,978300174
616072,3,2081,4.0,978298504
961113,5,288,2.0,978246585
670609,6,597,5.0,978239019
...,...,...,...,...
556135,6035,3146,1.0,956713640
933646,6036,2643,1.0,956755196
717654,6037,968,4.0,956801840
845930,6039,922,5.0,956758029


In [45]:
train = ratings.drop(test.index)

In [46]:
train

,user_id,movie_id,user_rating,timestamp
486815,1,3186,4.0,978300019
17504,1,1270,5.0,978300055
918576,1,1022,5.0,978300055
985165,1,1721,4.0,978300055
52019,1,2340,3.0,978300103
...,...,...,...,...
346679,6040,232,5.0,997454398
940072,6040,2917,4.0,997454429
610722,6040,1921,4.0,997454464
624387,6040,1784,3.0,997454464


In [47]:
print(train.shape)
print(test.shape)

(995912, 4)
(4297, 4)


In [48]:
from surprise import Dataset, Reader

reader = Reader(
    rating_scale=(1, 5)
)

In [49]:
train_data = Dataset.load_from_df(
    train[["user_id", "movie_id", "user_rating"]],
    reader
)

In [50]:
train_data

In [51]:
from surprise import SVD

model = SVD(
    n_factors=100,
    n_epochs=20,
    lr_all=0.005,
    reg_all=0.02,
    random_state=42
)

In [52]:
trainset = train_data.build_full_trainset()

model.fit(trainset)

In [53]:
# Recommendation-function

In [54]:
def collaborative_recommend(user_id, n=10):

    user_rated_movies = (
        ratings.groupby("user_id")["movie_id"]
        .apply(set)
        .to_dict()
    )

    rated_movies = user_rated_movies.get(user_id, set())

    all_movies = movies["movie_id"].unique()

    predictions = []

    for movie_id in all_movies:

        if movie_id not in rated_movies:

            pred = model.predict(
                user_id,
                movie_id
            )

            predictions.append(
                (movie_id, pred.est)
            )

    predictions.sort(
        key=lambda x: x[1],
        reverse=True
    )

    top_movies = predictions[:n]

    result = pd.DataFrame(
        top_movies,
        columns=["movie_id", "predicted_rating"]
    )

    result = result.merge(
        movies[
            ["movie_id", "movie_title", "movie_genres", "poster_url"]
        ],
        on="movie_id",
        how="left"
    )

    return result

In [55]:
collaborative_recommend(110, 10)

,movie_id,predicted_rating,movie_title,movie_genres,poster_url
0,2019,4.876474,Seven Samurai (The Magnificent Seven) (Shichin...,"0, 7",https://image.tmdb.org/t/p/w500/8OKmBV5BUFzmoz...
1,858,4.843522,"Godfather, The (1972)","0, 5, 7",https://image.tmdb.org/t/p/w500/3bhkrj58Vtu7en...
2,318,4.843522,"Shawshank Redemption, The (1994)",7,https://image.tmdb.org/t/p/w500/9cqNxx0GxF0bfl...
3,593,4.725739,"Silence of the Lambs, The (1991)","7, 16",https://image.tmdb.org/t/p/w500/cU6gd6i1tnXnPz...
4,527,4.711494,Schindler's List (1993),"7, 18",https://image.tmdb.org/t/p/w500/sF1U4EUQS8YHUY...
5,3022,4.700700,"General, The (1927)",4,https://image.tmdb.org/t/p/w500/qZKKwXyZ92K0mI...
6,1136,4.695086,Monty Python and the Holy Grail (1974),4,https://image.tmdb.org/t/p/w500/8AVb7tyxZRsbKJ...
7,260,4.691689,Star Wars: Episode IV - A New Hope (1977),"0, 1, 8, 15",https://image.tmdb.org/t/p/w500/6FfCtAuVAW8XJj...
8,1196,4.688436,Star Wars: Episode V - The Empire Strikes Back...,"0, 1, 7, 15, 18",https://image.tmdb.org/t/p/w500/9ldJDAUVPpFyod...
9,750,4.680646,Dr. Strangelove or: How I Learned to Stop Worr...,"15, 18",https://image.tmdb.org/t/p/w500/6x7MzQ6BOMlRza...


In [56]:
def has_rating_history(user_id):

    return (ratings["user_id"] == user_id).any()

In [57]:
def recommend_movies(user_id, n=10):

    if has_rating_history(user_id):

        return collaborative_recommend(
            user_id,
            n
        )

    else:

        return popular_movies.head(n)

In [58]:
user_id = 9999999

recommend_movies(user_id, 10)

,movie_id,avg_rating,count_rating,weighted_score,movie_title,movie_genres,poster_url
0,318,4.554558,2227,4.229925,"Shawshank Redemption, The (1994)",7,https://image.tmdb.org/t/p/w500/9cqNxx0GxF0bfl...
1,260,4.453694,2991,4.215501,Star Wars: Episode IV - A New Hope (1977),"0, 1, 8, 15",https://image.tmdb.org/t/p/w500/6FfCtAuVAW8XJj...
2,858,4.524966,2223,4.207205,"Godfather, The (1972)","0, 5, 7",https://image.tmdb.org/t/p/w500/3bhkrj58Vtu7en...
3,527,4.510417,2304,4.204639,Schindler's List (1993),"7, 18",https://image.tmdb.org/t/p/w500/sF1U4EUQS8YHUY...
4,1198,4.477725,2514,4.199097,Raiders of the Lost Ark (1981),"0, 1",https://image.tmdb.org/t/p/w500/ceG9VzoRAVGwiv...
5,50,4.517106,1783,4.145979,"Usual Suspects, The (1995)","5, 16",https://image.tmdb.org/t/p/w500/rWbsxdwF9qQzpT...
6,2762,4.406263,2459,4.139179,"Sixth Sense, The (1999)",16,https://image.tmdb.org/t/p/w500/vOyfUXNFSnaTk7...
7,2858,4.317386,3428,4.128147,American Beauty (1999),"4, 7",https://image.tmdb.org/t/p/w500/s5PXkDqS8W3K4w...
8,593,4.351823,2578,4.106356,"Silence of the Lambs, The (1991)","7, 16",https://image.tmdb.org/t/p/w500/cU6gd6i1tnXnPz...
9,2028,4.337354,2653,4.100450,Saving Private Ryan (1998),"0, 7, 18",https://image.tmdb.org/t/p/w500/uqx37cS8cpHg8U...


In [65]:
from surprise import accuracy

predictions = model.test(
    test[["user_id", "movie_id", "user_rating"]]
    .itertuples(index=False, name=None)
)

accuracy.rmse(predictions)

RMSE: 0.9082


np.float64(0.9081595645969319)

In [66]:
import pickle

with open("movie_svd_model.pkl", "wb") as file:
    pickle.dump(model, file)

In [67]:
import os

print(os.path.exists("movie_svd_model.pkl"))

True


In [68]:
import pickle

with open("movie_svd_model.pkl", "rb") as file:
    loaded_model = pickle.load(file)

In [69]:
prediction = loaded_model.predict(1, 50)

print(prediction.est)

4.454862940976998
